# Training a Model from Custom Folders

This notebook demonstrates how to load images from a custom folder structure (`dataset/target` and `dataset/background`) to train a robust classification model.

This approach is highly recommended for games like TDS (Tower Defense Simulator) where targets spawn at different positions, under varying camera angles, lighting conditions, or scale.

In [ ]:
# Install required libraries if not already installed
!pip install torch torchvision pillow matplotlib numpy opencv-python

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

## 1. Load the Dataset using ImageFolder

We use PyTorch's `ImageFolder` to load our organized folders. 
To make the model robust to various camera angles, zoom/scales, and positions in TDS, we apply data augmentations:
- `RandomRotation(15)`: Simulates camera angle rotations.
- `RandomHorizontalFlip()`: Simulates camera orbit/flips.
- `ColorJitter`: Simulates different lighting or environmental changes.

In [ ]:
dataset_dir = 'dataset'

train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(15),       # Rotates up to 15 degrees to handle camera angle changes
    transforms.RandomHorizontalFlip(),   # Flips horizontally
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Handles lighting/time-of-day changes
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

if not os.path.exists(dataset_dir):
    print(f"Error: '{dataset_dir}' directory not found! Please run the dataset generator first.")
else:
    full_dataset = datasets.ImageFolder(root=dataset_dir, transform=train_transforms)
    print(f"Loaded {len(full_dataset)} images from folder structure.")
    print(f"Classes found: {full_dataset.classes} -> mapping: {full_dataset.class_to_idx}")

## 2. Visualize Samples from Folders
Let's see some of the images we loaded from the folder structure.

In [ ]:
# Helper to denormalize and show image
def imshow(img, title=None):
    img = img * 0.5 + 0.5     # denormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title)
    plt.axis('off')

# Display first few images
plt.figure(figsize=(10, 5))
num_samples = min(8, len(full_dataset)) if 'full_dataset' in locals() and len(full_dataset) > 0 else 0
for idx in range(num_samples):
    img, label = full_dataset[idx]
    class_name = full_dataset.classes[label]
    plt.subplot(2, 4, idx + 1)
    imshow(img, f"{class_name} ({label})")
plt.tight_layout()
plt.show()

## 3. Split Dataset & Create DataLoaders
We split the dataset into 80% training and 20% validation.

In [ ]:
if 'full_dataset' in locals() and len(full_dataset) > 0:
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

    print(f"Training set: {len(train_dataset)} samples")
    print(f"Validation set: {len(val_dataset)} samples")
else:
    print("Please add some images in 'dataset/target' and 'dataset/background' folders first!")

## 4. Define the CNN Model
We use a lightweight CNN classifier suitable for object detection & categorization.

In [ ]:
class GameObjectCNN(nn.Module):
    def __init__(self):
        super(GameObjectCNN, self).__init__()
        self.features = nn.Sequential(
            # Input: 3x64x64 -> Output: 16x32x32
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Input: 16x32x32 -> Output: 32x16x16
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Input: 32x16x16 -> Output: 64x8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)  # 2 classes: 0 (background), 1 (target)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = GameObjectCNN()
print(model)

## 5. Train the Model
We will run the training loop once you populate the dataset folders.

In [ ]:
if 'train_loader' in locals():
    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = GameObjectCNN().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    epochs = 15

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct / total
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
        epoch_val_loss = val_loss / len(val_dataset)
        epoch_val_acc = val_correct / val_total if val_total > 0 else 0
        
        train_losses.append(epoch_loss)
        val_losses.append(epoch_val_loss)
        
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
else:
    print("Train loader not defined. Make sure you add images to dataset/ folders and run previous cells.")

## 6. Plot Metrics & Save Model
Plot training loss vs validation loss and save the model weights.

In [ ]:
if 'train_losses' in locals():
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.title('Loss Curves')
    plt.legend()
    plt.show()

    # Save weights
    torch.save(model.state_dict(), 'game_target_detector.pth')
    print("Model saved successfully to 'game_target_detector.pth'")
else:
    print("No model trained yet to save.")